## Dataset and Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, auc, classification_report)
from sklearn.cluster import KMeans


In [ ]:
df = pd.read_csv("data/heart_disease.csv")
df.head()

## Dataset Description

###Basic info, data types, and missing values

In [ ]:
df.info()

### Features count

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Input Features:", df.shape[1]-1)
print()
print("Target variable 'num' value counts (0 = no disease, 1-4 = increasing severity):")
print(df['num'].value_counts().sort_index())

### Categorical features count

In [ ]:
categorical_features = df.select_dtypes(include=["object"]).columns
print("Categorical features:")
print(categorical_features.tolist())
for column in categorical_features:
    print("\n", column)
    print(df[column].value_counts(dropna=False))

###Imbalanced dataset check

In [ ]:
plt.figure(figsize=(7, 5))
counts = df['num'].value_counts().sort_index()
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
for i, v in enumerate(counts.values):
    ax.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.xlabel("num (heart disease severity class)")
plt.ylabel("Number of instances")
plt.title("Class Distribution of Target Variable 'num' (0=No disease, 1-4=Severity)")
plt.tight_layout()
plt.show()

In [ ]:
df['target'] = (df['num'] > 0).astype(int)
print(df['target'].value_counts())

plt.figure(figsize=(5, 5))
counts_b = df['target'].value_counts().sort_index()
ax = sns.barplot(x=['No Disease (0)', 'Disease (1)'], y=counts_b.values,
                  hue=['No Disease (0)', 'Disease (1)'], palette="Set2", legend=False)
for i, v in enumerate(counts_b.values):
    ax.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.ylabel("Number of instances")
plt.title("Class Distribution of Binary Target (Disease Presence)")
plt.tight_layout()
plt.show()

## Exploratory Data Analysis(EDA)

In [ ]:
plt.figure(figsize=(7, 5))
sns.histplot(data=df, x='age', hue='target', kde=True, palette=['#4C72B0', '#DD8452'], multiple='stack')
plt.title("Disease Presence vs Age")
plt.xlabel("Age")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(data=df, x='sex', hue='target', palette=['#4C72B0', '#DD8452'])
plt.title("Disease Presence vs Sex")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='cp', hue='target', palette=['#4C72B0', '#DD8452'])
plt.title("Disease Presence vs Chest Pain Type ")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Fault Check

### Missing Value count and percentage

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df)) * 100
})

missing

### Numerical Features' Scales

In [ ]:
numerical_features = df.select_dtypes(
    include=["int64", "float64"]
).columns

print(numerical_features.tolist())


df[numerical_features].hist(
    figsize=(15, 10),
    bins=20
)

plt.tight_layout()
plt.show()

## Solution for faults

### Dropping columns

In [ ]:
drop_cols = ['id', 'dataset', 'num', 'ca', 'thal', 'slope']
df_clean = df.drop(columns=drop_cols)
print("Remaining columns:", df_clean.columns.tolist())

### Impute: Categorial columns(boolean-like text)

In [ ]:
def to_bool(x):
    if isinstance(x, str):
        return 1 if x.strip().lower() == 'true' else 0
    if isinstance(x, bool):
        return int(x)
    return x

df_clean['fbs'] = df_clean['fbs'].apply(to_bool)
df_clean['exang'] = df_clean['exang'].apply(to_bool)
print(df_clean[['fbs', 'exang']].head())

###Impute remaining missing values

In [ ]:
num_impute_cols = ['trestbps', 'chol', 'thalch', 'oldpeak']
for c in num_impute_cols:
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())

cat_impute_cols = ['fbs', 'restecg', 'exang']
for c in cat_impute_cols:
    df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])

print("Remaining missing values:", df_clean.isnull().sum().sum())

missing = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Percentage": (df_clean.isnull().sum() / len(df)) * 100
})

missing

### Encoded categorial variables(one hot encoding)

In [ ]:
df_clean = pd.get_dummies(df_clean, columns=['sex', 'cp', 'restecg'], drop_first=True)

df_clean['fbs'] = df_clean['fbs'].astype(int)
df_clean['exang'] = df_clean['exang'].astype(int)

for c in df_clean.columns:
    if df_clean[c].dtype == bool:
        df_clean[c] = df_clean[c].astype(int)

print(df_clean.dtypes)
print("Final shape:", df_clean.shape)

###Final Features Count

In [ ]:
X = df_clean.drop(columns=['target'])
y = df_clean['target']

feature_names = X.columns.tolist()
print("Number of final input features:", len(feature_names))
print("Features used:", feature_names)

### Correlation heatmap of all input and output features

In [ ]:
corr = df_clean.corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation Heatmap of Cleaned Input and Output Features")
plt.tight_layout()
plt.show()


## Dataset Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print()
print("Training set class balance:")
print(y_train.value_counts())
print()
print("Test set class balance:")
print(y_test.value_counts())

### Feature scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. X_train_scaled shape:", X_train_scaled.shape)
print(X_train_scaled[:5])

##Model Training and Testing (Supervised)

### Training Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000,
                                           random_state=42, early_stopping=True),
}

results = {}
roc_data = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    results[name] = {
        "accuracy": acc, "precision": prec, "recall": rec,
        "auc": roc_auc, "confusion_matrix": cm.tolist()
    }
    roc_data[name] = {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": roc_auc}

    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  AUC: {roc_auc:.4f}")
    print(cm)
    print()

###  Unsupervised Learning: Kmeans clustering

In [ ]:
X_all = np.vstack([X_train_scaled, X_test_scaled])

# Elbow method to help choose k
inertias = []
K_range = range(1, 50)
fitted_models = {}   # store each fitted model so we don't refit later

for k_val in K_range:
    km = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    km.fit(X_all)
    inertias.append(km.inertia_)
    fitted_models[k_val] = km   # keep it around

plt.figure(figsize=(7, 5))
plt.plot(list(K_range), inertias, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method for Optimal k")
plt.tight_layout()
plt.show()

In [ ]:
k = 5

kmeans = fitted_models[k]
cluster_labels = kmeans.labels_


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all)

plt.figure(figsize=(7, 5))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_labels,
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters")
plt.show()

## Model Selection/Comparison analysis

### Accuracy Comparison

In [ ]:
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=names, y=accs, hue=names, palette="mako", legend=False)
for i, v in enumerate(accs):
    ax.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Model Comparison: Test Accuracy")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Precision and Recall Comparison

In [ ]:
metrics_df = pd.DataFrame({
    "Model": names,
    "Precision": [results[n]['precision'] for n in names],
    "Recall": [results[n]['recall'] for n in names],

})
metrics_melt = metrics_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_melt, x="Model", y="Score", hue="Metric", palette="Set2")
plt.ylim(0, 1.05)
plt.title("Precision and Recall Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()
for i, name in enumerate(names):
    cm = np.array(results[name]['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'], cbar=False)
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
axes[-1].axis('off')
plt.suptitle("Confusion Matrices for All Models", fontsize=14)
plt.tight_layout()
plt.show()

### AUC score, ROC curve for each model

In [ ]:
plt.figure(figsize=(7, 7))
for name in names:
    fpr = roc_data[name]['fpr']
    tpr = roc_data[name]['tpr']
    roc_auc = roc_data[name]['auc']
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Models")
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### Summary Table

In [ ]:
summary_df = pd.DataFrame(results).T[['accuracy', 'precision', 'recall', 'auc']]
summary_df = summary_df.round(3)
summary_df